# exp6_social_discount — horizon, independent discounts

Le flux empathique est actualisé avec son propre $\gamma_o$, à $w_e$ = 0,5 et γ = 0,99,
avec le DQN à deux têtes (méthode journal, *Independent discounts* ; dérivation dans
`analysis_01_social_discount_factor.ipynb`).

Baisser $\gamma_o$ raccourcit l'empathie, mais réduit aussi son poids total. Chaque niveau
est donc apparié à un contrôle à horizon unique dont le $w_e'$ donne la même part empathique
totale : le contraste sépare « moins d'empathie » de « une empathie plus courte ».

Analyses : $r_s$ de Spearman sur $\gamma_o$, puis chaque niveau contre son contrôle
(Mann–Whitney, Bonferroni sur les six paires). $\gamma_o = \gamma$ est la référence et
n'a pas de contrôle.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

EXPERIMENT = "exp6_social_discount"
RESULTS_DIR = Path("..") / "csv_results" / EXPERIMENT
LAST_N_EVAL = 10
ALPHA_LEVEL = 0.05
METRICS = ["depletion_step", "gini_meals", "total_meals"]

## 1. Chargement et contrôles

In [ ]:
def load_runs(folder):
    """evaluation episodes of the finished runs of one experiment folder."""
    files = sorted(Path(folder).glob("*_episode_summary.csv"))
    if not files:
        raise FileNotFoundError(f"no *_episode_summary.csv under {Path(folder).resolve()}")
    frames = [pd.read_csv(p).assign(source_file=p.name) for p in files]
    df = pd.concat(frames, ignore_index=True)
    last = df[df.phase == "train"].groupby("source_file").episode.max()
    unfinished = last.index[last < last.max()]
    if len(unfinished):
        print(f"{len(unfinished)} unfinished run(s) dropped: {list(unfinished)}")
        df = df[~df.source_file.isin(unfinished)]
    df["depletion_step"] = df.total_steps
    df["censored"] = (df.total_steps >= df.max_steps) & (df.resource_remaining > 0)
    df["total_meals"] = df[[f"meals_{i}" for i in range(6)]].sum(axis=1)
    return df[df.phase == "eval"].copy()


def run_values(ev, keys):
    """one value per run: its mean over the last LAST_N_EVAL evaluation episodes."""
    tail = ev.sort_values("episode").groupby(keys + ["seed"], observed=True).tail(LAST_N_EVAL)
    return (tail.groupby(keys + ["seed"], observed=True)
                .agg(depletion_step=("depletion_step", "mean"), gini_meals=("gini_meals", "mean"),
                     total_meals=("total_meals", "mean"), censored=("censored", "mean"))
                .reset_index())


ev = load_runs(RESULTS_DIR)
print("duplicated evaluation rows:", int(ev.duplicated(["condition", "seed", "episode"]).sum()))
runs = run_values(ev, ["condition", "alpha"])
# the labels carry the level: D_res_go0950 and its control D_res_alphamatch_go0950
runs["gamma_o"] = runs.condition.str.extract(r"_go(\d{4})")[0].astype(int) / 1000
runs["control"] = runs.condition.str.contains("alphamatch")
display(runs.groupby(["gamma_o", "control"]).agg(alpha=("alpha", "first"), runs=("seed", "nunique"),
                                                 censored=("censored", "mean"),
                                                 **{m: (m, "mean") for m in METRICS}).round(3))

## 2. Tendance sur $\gamma_o$ (manipulation seule)

In [ ]:
def spearman_ci(rho: float, n: int, conf: float = 0.95) -> tuple[float, float]:
    """Fisher-z CI for Spearman's rho, with the Bonett-Wright variance correction."""
    if n < 6 or not np.isfinite(rho) or abs(rho) >= 1:
        return (np.nan, np.nan)
    delta = stats.norm.ppf(1 - (1 - conf) / 2) * np.sqrt((1 + rho ** 2 / 2) / (n - 3))
    return float(np.tanh(np.arctanh(rho) - delta)), float(np.tanh(np.arctanh(rho) + delta))


manipulation = runs[~runs.control]
rows = []
for metric in METRICS:
    rho_s, p_s = stats.spearmanr(manipulation.gamma_o, manipulation[metric])
    low, high = spearman_ci(rho_s, len(manipulation))
    rows.append({"metric": metric, "n": len(manipulation), "r_s": rho_s, "ci_low": low, "ci_high": high, "p": p_s})
display(pd.DataFrame(rows).round(4))

## 3. Chaque niveau contre son contrôle apparié

In [ ]:
def hedges_g(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled * (1 - 3 / (4 * (na + nb) - 9))


controls = runs[runs.control]
pairs = sorted(controls.gamma_o.unique())
rows = []
for metric in METRICS:
    for level in pairs:
        a = manipulation.loc[manipulation.gamma_o == level, metric].to_numpy()
        b = controls.loc[controls.gamma_o == level, metric].to_numpy()
        if len(a) < 2 or len(b) < 2:
            continue
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        rows.append({"metric": metric, "gamma_o": level, "manipulation": a.mean(), "control": b.mean(),
                     "g": hedges_g(a, b), "p": p, "p_bonferroni": min(1.0, p * len(pairs))})
contrasts = pd.DataFrame(rows)
contrasts["significant"] = contrasts.p_bonferroni < ALPHA_LEVEL
display(contrasts.round(4))